# Rib Waveguide: Proving `inner_product(...)` Matches Tidy3D Dot

This notebook compares, on the same Tidy3D mode fields:
- Tidy3D native overlap matrix: `ModeSolverData.outer_dot(..., conjugate=...)`
- MEOW overlap matrix using `mw.inner_product(..., conjugate=...)`

If the implementation is equivalent, the matrix difference should be near numerical precision.


In [ ]:
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import tidy3d as td
from tidy3d.plugins.mode import ModeSolver

import meow as mw

In [ ]:
# Parameters (aligned with rib_waveguide.ipynb)
wl = 1.0
freq = td.C_0 / (wl * 1e-6)
widths = [0.8, 1.0]
num_modes = 100

t_slab = 0.2
t_soi = 1.5
t_ox = 0.2
n_Si = 3.4
n_SiO2 = 1.4
w_sim = 3.0
h_sim = 2.5
y_bot = -0.5
y_center = y_bot + h_sim / 2

dl = w_sim / 100

si_td = td.Medium(permittivity=n_Si**2)
sio2_td = td.Medium(permittivity=n_SiO2**2)
env_mw = mw.Environment(wl=wl, T=25.0)

In [ ]:
def make_tidy3d_structures(width):
    return [
        td.Structure(
            geometry=td.Box(
                center=(0, 0.5 * (y_bot + (t_slab + t_ox)), 0),
                size=(w_sim, (t_slab + t_ox) - y_bot, td.inf),
            ),
            medium=sio2_td,
        ),
        td.Structure(
            geometry=td.Box(
                center=(0, 0.5 * ((t_slab + t_ox) + (t_soi + t_ox)), 0),
                size=(width + t_ox, t_soi - t_slab, td.inf),
            ),
            medium=sio2_td,
        ),
        td.Structure(
            geometry=td.Box(center=(0, 0.5 * t_slab, 0), size=(w_sim, t_slab, td.inf)),
            medium=si_td,
        ),
        td.Structure(
            geometry=td.Box(
                center=(0, 0.5 * (t_slab + t_soi), 0),
                size=(width, t_soi - t_slab, td.inf),
            ),
            medium=si_td,
        ),
    ]


def solve_tidy3d_mode_data(width):
    sim = td.Simulation(
        size=(w_sim, h_sim, 0),
        center=(0, y_center, 0),
        grid_spec=td.GridSpec.uniform(dl=dl),
        structures=make_tidy3d_structures(width),
        run_time=1e-12,
        boundary_spec=td.BoundarySpec(
            x=td.Boundary.pml(),
            y=td.Boundary.pml(),
            z=td.Boundary.periodic(),
        ),
    )
    ms = ModeSolver(
        simulation=sim,
        plane=td.Box(center=(0, y_center, 0), size=(w_sim, h_sim, 0)),
        mode_spec=td.ModeSpec(num_modes=num_modes),
        freqs=[freq],
    )
    return ms.solve()

In [ ]:
def centers_to_boundaries(c):
    c = np.asarray(c, dtype=float)
    if c.size < 2:
        raise ValueError("Need at least 2 center points.")
    edges = np.empty(c.size + 1, dtype=float)
    edges[1:-1] = 0.5 * (c[:-1] + c[1:])
    edges[0] = c[0] - 0.5 * (c[1] - c[0])
    edges[-1] = c[-1] + 0.5 * (c[-1] - c[-2])
    return edges


def _mode_from_tidy_fields(ex, ey, hx, hy, mesh):
    z = np.zeros_like(ex)
    cs = mw.CrossSection(structures=[], mesh=mesh, env=env_mw)
    return mw.Mode(
        neff=0.0 + 0.0j,
        cs=cs,
        Ex=np.asarray(ex),
        Ey=np.asarray(ey),
        Ez=z,
        Hx=np.asarray(hx),
        Hy=np.asarray(hy),
        Hz=z,
    )


def tidy3d_modes_as_meow_modes(mode_data):
    # Use the same tangential fields used by Tidy3D dot/outer_dot internals.
    fields = mode_data._colocated_tangential_fields

    e1 = np.asarray(fields["Ex"].isel(f=0).values)
    e2 = np.asarray(fields["Ey"].isel(f=0).values)
    h1 = np.asarray(fields["Hx"].isel(f=0).values)
    h2 = np.asarray(fields["Hy"].isel(f=0).values)

    x_centers = np.asarray(fields["Ex"].coords["x"].values, dtype=float)
    y_centers = np.asarray(fields["Ex"].coords["y"].values, dtype=float)
    mesh = mw.Mesh2D(
        x=centers_to_boundaries(x_centers),
        y=centers_to_boundaries(y_centers),
        num_pml=(0, 0),
    )

    modes = [
        _mode_from_tidy_fields(e1[..., i], e2[..., i], h1[..., i], h2[..., i], mesh)
        for i in range(e1.shape[-1])
    ]
    return modes


def meow_overlap_matrix(modes, op):
    n = len(modes)
    M = np.zeros((n, n), dtype=np.complex128)
    for i, mi in enumerate(modes):
        for j, mj in enumerate(modes):
            M[i, j] = op(mi, mj)
    return M

In [ ]:
def compare_one_width(label, width):
    md = solve_tidy3d_mode_data(width)

    M_td = np.asarray(md.outer_dot(md, conjugate=False).isel(f=0).values)
    M_td_conj = np.asarray(md.outer_dot(md, conjugate=True).isel(f=0).values)

    modes_td = tidy3d_modes_as_meow_modes(md)
    M_mw_sym = meow_overlap_matrix(
        modes_td,
        lambda a, b: mw.inner_product(a, b, conjugate=False, ignore_pml=False),
    )
    M_mw_sym_conj = meow_overlap_matrix(
        modes_td,
        lambda a, b: mw.inner_product(a, b, conjugate=True, ignore_pml=False),
    )

    print(f"\n{label} (w={width})")
    print(
        f"||M_meow_sym - M_tidy3d||_F (non-conj) = {np.linalg.norm(M_mw_sym - M_td):.3e}"
    )
    print(
        f"max|Δ| (non-conj)                     = {np.max(np.abs(M_mw_sym - M_td)):.3e}"
    )
    print(
        f"||M_meow_sym - M_tidy3d||_F (conj)    = {np.linalg.norm(M_mw_sym_conj - M_td_conj):.3e}"
    )
    print(
        f"max|Δ| (conj)                         = {np.max(np.abs(M_mw_sym_conj - M_td_conj)):.3e}"
    )

    # sanity checks for scale/shape
    I = np.eye(M_td.shape[0], dtype=np.complex128)
    print(f"||M_tidy3d - I||_F (non-conj)         = {np.linalg.norm(M_td - I):.3e}")
    print(f"||M_meow_sym - I||_F (non-conj)       = {np.linalg.norm(M_mw_sym - I):.3e}")

In [ ]:
compare_one_width("Left", widths[0])
compare_one_width("Right", widths[1])